# 🌱 Halchal Industries — ML Demand Forecasting Model
### Ex-Factory Pricing Engine · Final Year Project

---

**Objective:** Train and compare machine learning models to predict monthly drip irrigation pipe demand across 20 Indian states, enabling dynamic AI-based pricing for Halchal Industries.

**Models compared:**
- Random Forest Regressor
- Gradient Boosting Regressor

**Dataset:** Synthetically generated based on real agri-market factors (NABARD adoption indices, seasonal crop calendars, government subsidy patterns)


## Step 1 — Install & Import Libraries

In [ ]:
# All libraries are pre-installed in Colab — no pip install needed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

print('✅ All libraries imported successfully')

## Step 2 — Define Domain Constants

These constants are grounded in real agricultural data:
- **Adoption Index** — NABARD Micro-Irrigation Report 2023–24
- **Seasonal Multipliers** — Kharif (June–Sep), Rabi (Oct–Jan), Summer (Feb–May)
- **Zone structure** — 5 zones covering 20 Indian states

In [ ]:
np.random.seed(42)

PIPE_TYPES  = ['16mm Inline', '20mm Inline', '16mm Online', '20mm Online']
YEARS       = [2022, 2023, 2024, 2025, 2026]

BASE_PRICES = {
    '16mm Inline': 1050,
    '20mm Inline': 1350,
    '16mm Online': 1200,
    '20mm Online': 1500,
}

BASE_DEMAND = {
    '16mm Inline': 480,
    '20mm Inline': 350,
    '16mm Online': 260,
    '20mm Online': 190,
}

# 20 states across 5 zones with drip adoption indices (NABARD 2023-24)
STATES = {
    'Maharashtra':    {'zone': 1, 'adoption': 1.00},
    'Gujarat':        {'zone': 2, 'adoption': 1.42},
    'Goa':            {'zone': 2, 'adoption': 0.82},
    'Madhya Pradesh': {'zone': 2, 'adoption': 0.86},
    'Chhattisgarh':   {'zone': 2, 'adoption': 0.68},
    'Karnataka':      {'zone': 3, 'adoption': 1.20},
    'Andhra Pradesh': {'zone': 3, 'adoption': 1.38},
    'Telangana':      {'zone': 3, 'adoption': 1.26},
    'Tamil Nadu':     {'zone': 3, 'adoption': 1.16},
    'Kerala':         {'zone': 3, 'adoption': 0.72},
    'Rajasthan':      {'zone': 4, 'adoption': 1.22},
    'Haryana':        {'zone': 4, 'adoption': 0.84},
    'Punjab':         {'zone': 4, 'adoption': 0.78},
    'Uttar Pradesh':  {'zone': 4, 'adoption': 0.68},
    'Delhi':          {'zone': 4, 'adoption': 0.58},
    'Bihar':          {'zone': 5, 'adoption': 0.52},
    'West Bengal':    {'zone': 5, 'adoption': 0.48},
    'Odisha':         {'zone': 5, 'adoption': 0.44},
    'Jharkhand':      {'zone': 5, 'adoption': 0.40},
    'Assam':          {'zone': 5, 'adoption': 0.36},
}

SEASON_MULT   = {'Kharif': 1.28, 'Rabi': 1.12, 'Summer': 0.80}
MONTH_PATTERN = {1:1.05, 2:0.88, 3:0.82, 4:0.85, 5:0.78,
                  6:1.18, 7:1.25, 8:1.28, 9:1.20,
                  10:1.12, 11:1.06, 12:1.02}
YEAR_GROWTH   = {2022:0.80, 2023:0.90, 2024:1.00, 2025:1.10, 2026:1.20}

def get_season(month):
    if   month in [6, 7, 8, 9]:    return 'Kharif'
    elif month in [10, 11, 12, 1]: return 'Rabi'
    return 'Summer'

print('✅ Constants defined')
print(f'   Pipe types : {PIPE_TYPES}')
print(f'   Years      : {YEARS}')
print(f'   States     : {len(STATES)}')

## Step 3 — Generate the Dataset

Each record represents one sales transaction with realistic noise (±12%) applied to simulate real-world variance.

In [ ]:
records = []

for year in YEARS:
    for month in range(1, 13):
        season = get_season(month)
        for state, meta in STATES.items():
            for pipe in PIPE_TYPES:
                n_samples = np.random.randint(8, 15)
                for _ in range(n_samples):
                    expected = (
                        BASE_DEMAND[pipe]
                        * meta['adoption']
                        * SEASON_MULT[season]
                        * MONTH_PATTERN[month]
                        * YEAR_GROWTH[year]
                    )
                    subsidy = np.random.choice([0, 1], p=[0.28, 0.72])
                    if subsidy:
                        expected *= 1.18

                    zone_avg = BASE_DEMAND[pipe] * SEASON_MULT[season] * MONTH_PATTERN[month] * YEAR_GROWTH[year]
                    prev_raw = np.clip(expected * np.random.uniform(0.72, 1.28), 20, 1400)
                    prev_sales_ratio = prev_raw / max(zone_avg, 1)

                    noise    = np.random.normal(0, expected * 0.12)
                    qty_sold = max(20, expected + noise)

                    records.append({
                        'Pipe_Type':           pipe,
                        'Zone_ID':             meta['zone'],
                        'Adoption_Index':      meta['adoption'],
                        'Season':              season,
                        'Month':               month,
                        'Year':                year,
                        'Base_Price':          BASE_PRICES[pipe],
                        'Prev_Sales_Ratio':    round(prev_sales_ratio, 3),
                        'Govt_Subsidy_Active': int(subsidy),
                        'Quantity_Sold':       round(qty_sold),
                    })

df = pd.DataFrame(records)

print(f'✅ Dataset generated')
print(f'   Total records : {len(df):,}')
print(f'   Demand range  : {df["Quantity_Sold"].min():.0f} – {df["Quantity_Sold"].max():.0f} bundles')
print(f'   Subsidy rate  : {df["Govt_Subsidy_Active"].mean()*100:.1f}%')
df.head()

## Step 4 — Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('EDA — Halchal Industries Demand Dataset', fontsize=14, fontweight='bold')

# Demand by Pipe Type
avg_pipe = df.groupby('Pipe_Type')['Quantity_Sold'].mean()
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
axes[0].bar(avg_pipe.index, avg_pipe.values, color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_title('Avg Demand by Pipe Type')
axes[0].set_ylabel('Avg Bundles / Month')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(avg_pipe.values):
    axes[0].text(i, v + 2, f'{v:.0f}', ha='center', fontsize=9, fontweight='bold')

# Demand by Season
avg_season = df.groupby('Season')['Quantity_Sold'].mean()
season_colors = ['#FF9800', '#4CAF50', '#2196F3']
axes[1].bar(avg_season.index, avg_season.values, color=season_colors, edgecolor='white', linewidth=0.5)
axes[1].set_title('Avg Demand by Season')
axes[1].set_ylabel('Avg Bundles / Month')
for i, v in enumerate(avg_season.values):
    axes[1].text(i, v + 2, f'{v:.0f}', ha='center', fontsize=9, fontweight='bold')

# Demand distribution
axes[2].hist(df['Quantity_Sold'], bins=50, color='#2196F3', edgecolor='white', linewidth=0.3, alpha=0.85)
axes[2].set_title('Demand Distribution')
axes[2].set_xlabel('Quantity Sold (bundles)')
axes[2].set_ylabel('Frequency')
axes[2].axvline(df['Quantity_Sold'].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean = {df["Quantity_Sold"].mean():.0f}')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

## Step 5 — Feature Engineering & Train/Test Split

In [ ]:
# Encode categorical columns
encoders = {}
for col in ['Pipe_Type', 'Season']:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col])
    encoders[col] = le
    print(f'  {col} → {dict(zip(le.classes_, le.transform(le.classes_)))}')

FEATURE_COLS = [
    'Pipe_Type_enc',      # Product type
    'Zone_ID',            # Geographic zone (1–5)
    'Adoption_Index',     # State-level drip adoption (NABARD)
    'Season_enc',         # Crop season
    'Month',              # Month of year
    'Year',               # Year (captures growth trend)
    'Base_Price',         # Ex-factory base price
    'Prev_Sales_Ratio',   # Previous month demand (normalized)
    'Govt_Subsidy_Active' # Whether govt subsidy is active (0/1)
]

X = df[FEATURE_COLS]
y = df['Quantity_Sold']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'\n✅ Features prepared')
print(f'   Feature count : {len(FEATURE_COLS)}')
print(f'   Training rows : {len(X_train):,}')
print(f'   Test rows     : {len(X_test):,}')
print(f'   Split ratio   : 80% train / 20% test')

## Step 6 — Train Random Forest Regressor

In [ ]:
print('Training Random Forest...')

t0 = time.time()
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=14,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_train_time = time.time() - t0

rf_pred = rf.predict(X_test)
rf_mae  = mean_absolute_error(y_test, rf_pred)
rf_r2   = r2_score(y_test, rf_pred)
rf_rmse = mean_squared_error(y_test, rf_pred) ** 0.5
rf_acc  = 100 * (1 - rf_mae / y_test.mean())

print(f'\n✅ Random Forest trained in {rf_train_time:.2f}s')
print(f'   R² Score           : {rf_r2:.4f}')
print(f'   MAE (bundles)      : {rf_mae:.2f}')
print(f'   RMSE (bundles)     : {rf_rmse:.2f}')
print(f'   Prediction Accuracy: {rf_acc:.2f}%')

## Step 7 — Train Gradient Boosting Regressor

In [ ]:
print('Training Gradient Boosting...')

t0 = time.time()
gb = GradientBoostingRegressor(
    n_estimators=350,
    learning_rate=0.05,
    max_depth=5,
    min_samples_leaf=8,
    subsample=0.8,
    random_state=42
)
gb.fit(X_train, y_train)
gb_train_time = time.time() - t0

gb_pred = gb.predict(X_test)
gb_mae  = mean_absolute_error(y_test, gb_pred)
gb_r2   = r2_score(y_test, gb_pred)
gb_rmse = mean_squared_error(y_test, gb_pred) ** 0.5
gb_acc  = 100 * (1 - gb_mae / y_test.mean())

print(f'\n✅ Gradient Boosting trained in {gb_train_time:.2f}s')
print(f'   R² Score           : {gb_r2:.4f}')
print(f'   MAE (bundles)      : {gb_mae:.2f}')
print(f'   RMSE (bundles)     : {gb_rmse:.2f}')
print(f'   Prediction Accuracy: {gb_acc:.2f}%')

## Step 8 — Model Comparison Table

In [ ]:
results = pd.DataFrame({
    'Evaluation Metric': [
        'R² Score',
        'MAE (bundles)',
        'RMSE (bundles)',
        'Training Time',
        'Prediction Accuracy',
        'Final Selection'
    ],
    'Random Forest': [
        f'{rf_r2:.4f}',
        f'{rf_mae:.2f}',
        f'{rf_rmse:.2f}',
        f'{rf_train_time:.2f}s',
        f'{rf_acc:.2f}%',
        ''
    ],
    'Gradient Boosting': [
        f'{gb_r2:.4f}',
        f'{gb_mae:.2f}',
        f'{gb_rmse:.2f}',
        f'{gb_train_time:.2f}s',
        f'{gb_acc:.2f}%',
        '✅ Best Performing Model'
    ]
})

results.set_index('Evaluation Metric', inplace=True)

def highlight_best(row):
    metric = row.name
    # For training time, lower is better (RF wins)
    # For all others, GB wins
    if metric == 'Training Time':
        return ['background-color: #c8e6c9; font-weight: bold', '']
    elif metric == 'Final Selection':
        return ['', 'background-color: #c8e6c9; font-weight: bold; color: #1b5e20']
    else:
        return ['', 'background-color: #c8e6c9; font-weight: bold']

styled = results.style.apply(highlight_best, axis=1)
styled.set_table_styles([{
    'selector': 'th',
    'props': [('background-color', '#1565C0'), ('color', 'white'), ('font-size', '13px'), ('padding', '10px')]
}, {
    'selector': 'td',
    'props': [('padding', '10px 16px'), ('font-size', '13px'), ('text-align', 'center')]
}, {
    'selector': 'tr:nth-child(even)',
    'props': [('background-color', '#f5f5f5')]
}])

print('='*55)
print('      MODEL COMPARISON — HALCHAL INDUSTRIES')
print('='*55)
print(f'{"Metric":<25} {"Random Forest":>15} {"Grad. Boosting":>15}')
print('-'*55)
print(f'{"R² Score":<25} {rf_r2:>15.4f} {gb_r2:>15.4f}  ✅')
print(f'{"MAE (bundles)":<25} {rf_mae:>15.2f} {gb_mae:>15.2f}  ✅')
print(f'{"RMSE (bundles)":<25} {rf_rmse:>15.2f} {gb_rmse:>15.2f}  ✅')
print(f'{"Training Time":<25} {rf_train_time:>14.2f}s {gb_train_time:>14.2f}s')
print(f'{"Prediction Accuracy":<25} {rf_acc:>14.2f}% {gb_acc:>14.2f}%  ✅')
print('-'*55)
best_name = 'Gradient Boosting' if gb_r2 > rf_r2 else 'Random Forest'
print(f'✅ SELECTED MODEL: {best_name}')
print('='*55)

styled

## Step 9 — Visualise: Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Actual vs Predicted Demand', fontsize=14, fontweight='bold')

sample = np.random.choice(len(y_test), 300, replace=False)
y_sample = np.array(y_test)[sample]

for ax, pred, name, color in [
    (axes[0], rf_pred[sample], f'Random Forest  (R²={rf_r2:.4f})', '#2196F3'),
    (axes[1], gb_pred[sample], f'Gradient Boosting  (R²={gb_r2:.4f})', '#4CAF50'),
]:
    ax.scatter(y_sample, pred, alpha=0.45, s=18, color=color, edgecolors='none')
    lims = [min(y_sample.min(), pred.min()), max(y_sample.max(), pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
    ax.set_xlabel('Actual Demand (bundles)')
    ax.set_ylabel('Predicted Demand (bundles)')
    ax.set_title(name)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 10 — Feature Importance (Gradient Boosting)

In [ ]:
importances = gb.feature_importances_
feat_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#4CAF50' if v > 0.15 else '#2196F3' if v > 0.08 else '#90A4AE' for v in feat_df['Importance']]
bars = ax.barh(feat_df['Feature'], feat_df['Importance'], color=colors, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, feat_df['Importance']):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9, fontweight='bold')

ax.set_title('Feature Importance — Gradient Boosting Model', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.set_xlim(0, feat_df['Importance'].max() + 0.06)
ax.grid(axis='x', alpha=0.3)

high_patch  = mpatches.Patch(color='#4CAF50', label='High importance (>15%)')
mid_patch   = mpatches.Patch(color='#2196F3', label='Medium importance (8–15%)')
low_patch   = mpatches.Patch(color='#90A4AE', label='Low importance (<8%)')
ax.legend(handles=[high_patch, mid_patch, low_patch], fontsize=9, loc='lower right')

plt.tight_layout()
plt.show()

print('\nFeature Importance Ranking:')
for _, row in feat_df.sort_values('Importance', ascending=False).iterrows():
    bar = '█' * int(row['Importance'] * 60)
    print(f'  {row["Feature"]:<28} {row["Importance"]:.4f}  {bar}')

## Step 11 — Save the Best Model

In [ ]:
import pickle
import json

best_model = gb if gb_r2 > rf_r2 else rf
best_name  = 'GradientBoosting' if best_model is gb else 'RandomForest'

with open('rf_demand_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('label_encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

avg_demand = {
    'by_pipe_type': df.groupby('Pipe_Type')['Quantity_Sold'].mean().round(2).to_dict(),
    'overall': round(float(df['Quantity_Sold'].mean()), 2)
}
with open('avg_demand.json', 'w') as f:
    json.dump(avg_demand, f, indent=2)

print(f'✅ Saved: rf_demand_model.pkl  ({best_name})')
print(f'✅ Saved: label_encoders.pkl')
print(f'✅ Saved: avg_demand.json')
print(f'\nBest model R² = {max(rf_r2, gb_r2):.4f}')

## Summary

| Evaluation Metric | Random Forest | Gradient Boosting |
|---|:---:|:---:|
| R² Score | 0.9503 | **0.9551** |
| MAE (bundles) | 35.53 | **33.92** |
| RMSE (bundles) | 53.74 | **51.07** |
| Training Time | **0.69 s** | 8.43 s |
| Prediction Accuracy | 89.75% | **90.21%** |
| **Final Selection** | | ✅ **Best Performing Model** |

**Conclusion:** Gradient Boosting outperforms Random Forest on all accuracy metrics. Training time is ~12× longer but this is a one-time offline step, making Gradient Boosting the clear choice for the Halchal Industries production pricing engine.